# M03.P02 — MLP vs modelo clásico

Ruta esencial: comparar una red neuronal compacta con el baseline clásico sobre el mismo problema.
El objetivo no es que la red gane, sino justificar la decisión con evidencia.


In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("../assets/intent_requests.csv")
features = [
    "request_text",
    "channel",
    "business_unit",
    "language",
    "urgency",
    "requires_authoritative_sources",
]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    df[features], df["label"], test_size=0.20,
    random_state=42, stratify=df["label"]
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20,
    random_state=42, stratify=y_train_full
)

print("train / val / test:", len(X_train), len(X_val), len(X_test))


In [ ]:
categorical = [
    "channel", "business_unit", "language",
    "urgency", "requires_authoritative_sources"
]

def build_preprocessor():
    return ColumnTransformer([
        ("text", TfidfVectorizer(max_features=1000, ngram_range=(1, 2)), "request_text"),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ])

classic_pipeline = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("model", LogisticRegression(max_iter=1000)),
])

classic_pipeline.fit(X_train, y_train)
classic_pred = classic_pipeline.predict(X_test)

classic_f1 = f1_score(y_test, classic_pred, average="macro")
classic_acc = accuracy_score(y_test, classic_pred)
print("classic macro F1:", round(classic_f1, 4))
print("classic accuracy:", round(classic_acc, 4))


## Preparar los datos para la MLP

El preprocesamiento se ajusta únicamente con `train`. `validation` sirve para observar el entrenamiento y `test` queda reservado para la comparación final.


In [ ]:
neural_preprocessor = build_preprocessor()
Xtr = neural_preprocessor.fit_transform(X_train)
Xval = neural_preprocessor.transform(X_val)
Xte = neural_preprocessor.transform(X_test)

def dense_float32(x):
    if sparse.issparse(x):
        return x.toarray().astype("float32")
    return np.asarray(x, dtype="float32")

Xtr = dense_float32(Xtr)
Xval = dense_float32(Xval)
Xte = dense_float32(Xte)

label_encoder = LabelEncoder()
ytr = label_encoder.fit_transform(y_train)
yval = label_encoder.transform(y_val)
yte = label_encoder.transform(y_test)

train_loader = DataLoader(
    TensorDataset(torch.tensor(Xtr), torch.tensor(ytr, dtype=torch.long)),
    batch_size=64, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(torch.tensor(Xval), torch.tensor(yval, dtype=torch.long)),
    batch_size=64, shuffle=False
)

print("input_dim:", Xtr.shape[1], "classes:", len(label_encoder.classes_))


In [ ]:
sys.path.append("../enterprise-genai-assistant")
from src.neural_model import NeuralIntentMLP

# TODO 1: instancia la red
# model = NeuralIntentMLP(
#     input_dim=...,
#     num_classes=...,
#     hidden_dim=128,
# )

# TODO 2: elige loss y optimizer
# loss_fn = ...
# optimizer = ...


In [ ]:
def evaluate(loader):
    model.eval()
    losses, y_true, y_pred = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            losses.append(loss_fn(logits, yb).item())
            y_true.extend(yb.cpu().numpy())
            y_pred.extend(logits.argmax(dim=1).cpu().numpy())
    return float(np.mean(losses)), f1_score(y_true, y_pred, average="macro")

history = []
max_epochs = 8

for epoch in range(1, max_epochs + 1):
    model.train()
    for xb, yb in train_loader:
        # TODO 3: ciclo de entrenamiento
        # optimizer.zero_grad()
        # logits = ...
        # loss = ...
        # loss.backward()
        # optimizer.step()
        pass

    # Descomenta cuando completes el TODO.
    # val_loss, val_f1 = evaluate(val_loader)
    # history.append((epoch, val_loss, val_f1))
    # print(f"epoch={epoch:02d} val_loss={val_loss:.4f} val_f1={val_f1:.4f}")


In [ ]:
# TODO 4: evaluación final sobre test
# model.eval()
# with torch.no_grad():
#     neural_logits = model(torch.tensor(Xte, dtype=torch.float32))
# neural_pred = neural_logits.argmax(dim=1).cpu().numpy()
# neural_f1 = f1_score(yte, neural_pred, average="macro")
# neural_acc = accuracy_score(yte, neural_pred)

# print("classic:", classic_f1, classic_acc)
# print("neural :", neural_f1, neural_acc)


## Decisión

Responde:

1. ¿Qué modelo obtiene mejor macro F1?
2. ¿La diferencia es suficientemente grande para justificar una red neuronal?
3. ¿Qué coste operativo adicional introduce la MLP?
4. ¿Mantendrías el clásico, desplegarías el neuronal o seguirías experimentando?


## Ampliación opcional — exportar artefactos para M03.P06

Solo es necesaria si vas a realizar la integración completa del Enterprise GenAI Assistant v0.3.


In [ ]:
# Ejecuta esta celda solo si has completado la MLP y quieres continuar con P06.
# ART = Path("../enterprise-genai-assistant/artifacts")
# ART.mkdir(parents=True, exist_ok=True)
# joblib.dump(classic_pipeline, ART / "classic_router.joblib")
# joblib.dump(neural_preprocessor, ART / "neural_preprocessor.joblib")
# joblib.dump(label_encoder, ART / "neural_label_encoder.joblib")
# torch.save(model.state_dict(), ART / "neural_router.pt")
